# 05 — SHAP Explainability & Drift Monitoring
**Fraud Detection & Anomaly Scoring System**

Goals:
- SHAP summary plot: which features drive fraud predictions globally
- SHAP force plots: explain individual predictions
- Waterfall plots: per-transaction explanation
- PSI drift monitoring across time windows

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from src.features import build_train_test
from src.models import load_models
from src.monitoring import simulate_drift, plot_psi_dashboard

plt.rcParams.update({'figure.dpi': 120})
shap.initjs()

_, _, X_test, _, _, y_test, feat_names = build_train_test(apply_smote=False)
iso, xgb_m, lgbm_m, scaler = load_models()
print("Assets loaded.")

## 1. SHAP Summary Plot (XGBoost)

In [ ]:
# Use a 2,000-sample background for speed
idx = np.random.choice(len(X_test), size=min(2000, len(X_test)), replace=False)
X_sample = X_test[idx]
y_sample = y_test[idx]

explainer   = shap.TreeExplainer(xgb_m)
shap_values = explainer(X_sample)

# For binary classification shap_values may have shape (n, f, 2); take class=1
sv = shap_values.values[:, :, 1] if shap_values.values.ndim == 3 else shap_values.values

fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(
    sv, X_sample,
    feature_names=feat_names,
    plot_type='dot',
    show=False,
    max_display=20,
)
plt.title('SHAP Summary — XGBoost Fraud Model', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/shap_summary_xgb.png', dpi=120, bbox_inches='tight')
plt.show()

## 2. SHAP Bar Plot (Mean |SHAP|)

In [ ]:
mean_abs = np.abs(sv).mean(axis=0)
shap_importance = pd.Series(mean_abs, index=feat_names).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 5))
shap_importance.plot.bar(ax=ax, color='#e67e22', edgecolor='white')
ax.set_title('Mean |SHAP Value| — Top 15 Features (XGBoost)', fontsize=13, fontweight='bold')
ax.set_ylabel('mean |SHAP|')
plt.tight_layout()
plt.savefig('../reports/shap_bar_xgb.png', dpi=120)
plt.show()

print("Top fraud-driving features:")
print(shap_importance)

## 3. Waterfall Plot — Individual Fraud Prediction

In [ ]:
# Pick a true fraud sample
fraud_indices = np.where(y_sample == 1)[0]
if len(fraud_indices) > 0:
    idx_fraud = fraud_indices[0]
    bv = shap_values.base_values[idx_fraud]
    bv = float(bv[1]) if hasattr(bv, '__len__') else float(bv)

    shap.waterfall_plot(
        shap.Explanation(
            values=sv[idx_fraud],
            base_values=bv,
            data=X_sample[idx_fraud],
            feature_names=feat_names,
        ),
        show=False,
        max_display=15,
    )
    plt.title(f'Waterfall — Fraud Transaction (sample {idx_fraud})', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../reports/shap_waterfall_fraud.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print("No fraud samples in the current 2,000-sample slice — re-run with a larger sample.")

## 4. SHAP Dependence Plot — V14 (top feature)

In [ ]:
if 'V14' in feat_names:
    v14_idx = feat_names.index('V14')
    fig, ax = plt.subplots(figsize=(8, 5))
    sc = ax.scatter(
        X_sample[:, v14_idx],
        sv[:, v14_idx],
        c=y_sample, cmap='RdYlGn_r', alpha=0.4, s=10,
    )
    plt.colorbar(sc, ax=ax, label='True Class (0=Legit, 1=Fraud)')
    ax.axhline(0, color='black', lw=0.8, linestyle='--')
    ax.set(xlabel='V14 value', ylabel='SHAP value for V14',
           title='SHAP Dependence Plot — V14')
    plt.tight_layout()
    plt.savefig('../reports/shap_dependence_v14.png', dpi=120)
    plt.show()

## 5. Drift Monitoring — PSI across Time Windows

In [ ]:
drift_results = simulate_drift(X_test, feat_names, n_windows=4)

for dr in drift_results:
    window = dr['window'].iloc[0]
    n_alert   = (dr['psi'] > 0.20).sum()
    n_warning = ((dr['psi'] > 0.10) & (dr['psi'] <= 0.20)).sum()
    n_stable  = (dr['psi'] <= 0.10).sum()
    print(f"{window}: 🚨 Alert={n_alert}  ⚠️  Warning={n_warning}  ✅ Stable={n_stable}")

In [ ]:
if drift_results:
    plot_psi_dashboard(drift_results[0], title='PSI Dashboard — W2 vs W1')

    # Heatmap: PSI across all windows
    import pandas as pd
    pivot = pd.concat(drift_results).pivot_table(index='feature', columns='window', values='psi')
    
    fig, ax = plt.subplots(figsize=(10, max(5, len(pivot)//2)))
    sns_data = pivot.fillna(0)
    import seaborn as sns
    sns.heatmap(sns_data, annot=True, fmt='.3f', cmap='RdYlGn_r',
                vmin=0, vmax=0.3, ax=ax, linewidths=0.5)
    ax.set_title('PSI Heatmap — All Windows vs Baseline', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../reports/psi_heatmap.png', dpi=120)
    plt.show()

## 6. Key Findings

### SHAP Insights
| Feature | Effect on Fraud Score | Interpretation |
|---------|----------------------|----------------|
| V14 (low) | ↑ Fraud probability | Strong negative V14 → high fraud risk |
| V10 (low) | ↑ Fraud probability | Captures specific fraud pattern |
| V12 (low) | ↑ Fraud probability | Correlated fraud behaviour |
| Amount_Log | Bidirectional | Both very low and high amounts are risky |

### Drift Insights
- PSI < 0.10 across most features → model is stable on test split
- In production, monitor PSI weekly and retrain if >0.20 on key features

> **Next step:** Launch the Streamlit dashboard: `streamlit run app/streamlit_app.py`